In [ ]:
'''
Question 1: What is Pug? Why might developers prefer Pug over writing raw HTML for
server-rendered pages?
Answer:Pug (formerly known as Jade) is a high-performance template engine for Node.js. It allows developers to write dynamic HTML templates using simplified, indentation-based syntax.
Developers Prefer Pug over Raw HTML:
	Cleaner, Terse Syntax: Eliminates closing tags, angle brackets (<>), and verbose markup, drastically reducing boilerplates and code 	length.
	Indentation-Based Hierarchy: Enforces structured, clean formatting through strict indentation, improving readability.
	Built-in Control Flow & Logic: Allows direct execution of JavaScript expressions, loops (each), and conditionals (if/else) inside 	templates.
	Reusability & Components: Supports template inheritance (extends), block overrides (block), and reusable partials (include / mixin), 	making modular UI architecture simple to maintain.

Question 2: Initialize an Express server that uses Pug as the view engine. Create a route
`/welcome` that renders a Pug view `welcome.pug`, passing a variable `name`. The view should
greet the user by name. Show code for server setup and the Pug template.
Answer: Server Setup (app.js):
import express from 'express';
import path from 'path';

const app = express();

// Set Pug as the template engine
app.set('view engine', 'pug');
app.set('views', path.join(process.cwd(), 'views'));

// Route rendering the Pug template
app.get('/welcome', (req, res) => {
  res.render('welcome', { name: 'Alice' });
});

app.listen(3000);

Pug Template (views/welcome.pug):

doctype html
html(lang="en")
  head
    title Welcome Page
  body
    h1 Welcome, #{name}!
    p We're glad to have you back.

Question 3: Explain what a RESTful API is. Then list four HTTP methods (GET, POST,
PUT/PATCH, DELETE) and describe what each should do in a REST design for a resource
“/tasks”.
Answer: A RESTful API (Representational State Transfer) is an architectural style for designing networked applications. It relies on stateless communication over HTTP, manipulating resource representations (typically JSON or XML) via standardized HTTP request methods and predictable URI structures.

	HTTP Methods for the /tasks Resource:

		GET /tasks: Retrieves a list of all task resources (or details of a specific task if /tasks/:id). Should be idempotent and read-		only.

		POST /tasks: Creates a brand-new task resource using data sent in the request body.

		PUT / PATCH /tasks/:id: Updates an existing task. PUT replaces the entire resource representation, while PATCH partially updates 		specific fields.

		DELETE /tasks/:id: Removes the specified task resource from the server.

Question 4: Using Express, define routes for a Task resource: GET /tasks, POST /tasks,
PUT /tasks/:id, DELETE /tasks/:id. In each handler, just send JSON saying which
operation was called (no DB required yet). Write the Express route definitions.
Answer: import express from 'express';

const app = express();
app.use(express.json()); // Middleware to parse incoming JSON payloads

// GET /tasks - Retrieve tasks
app.get('/tasks', (req, res) => {
  res.json({ message: 'GET operation: Retreived all tasks' });
});

// POST /tasks - Create task
app.post('/tasks', (req, res) => {
  res.json({ message: 'POST operation: Created a new task' });
});

// PUT /tasks/:id - Update task
app.put('/tasks/:id', (req, res) => {
  res.json({ message: `PUT operation: Updated task with ID ${req.params.id}` });
});

// DELETE /tasks/:id - Delete task
app.delete('/tasks/:id', (req, res) => {
  res.json({ message: `DELETE operation: Removed task with ID ${req.params.id}` });
});

app.listen(3000);

Question 5: Describe in simple words what Passport.js does. Then sketch the flow of user login
using local strategy: user submits credentials, server checks them, and if valid, logs in (you can
assume a dummy password check). You don’t need to provide full code, just outline key steps
(Express + Passport + session or token).
Answer: Passport.js is an authentication middleware for Node.js. It handles user authentication requests by delegating tasks to modular strategies (such as username/password, OAuth with Google, or JWT tokens), keeping the login process separate from the main application logic.

	Local Strategy Authentication Flow (Sessions):

		Submit Credentials: The user submits a form containing their username and password via a POST /login request.
		Passport Interception: Passport receives the request and delegates the credentials to the LocalStrategy verify callback.
		Validate User: The server queries the database for the user by username. If found, it compares the provided password with the 			stored password hash (e.g., using bcrypt).
		Serialize User (Session Setup):
			If invalid: Authentication fails, returning a 401 Unauthorized response or redirecting to /login.
			If valid: Passport calls req.login() and executes passport.serializeUser(), which saves the user ID into the session 				store and attaches a session cookie to the response.
		Session Deserialization: On subsequent requests, Passport's session middleware reads the cookie, calls 						passport. deserializeUser() using the stored ID, and attaches the full user object to req.user.

Question 6: You want users to log in using their Google account (OAuth) instead of local
passwords. What are the high-level steps to implement this with Passport.js and Express?
Mention redirection, callback URL, and user data extraction.
Answer: High-Level Implementation Steps:

Google Developer Console Registration:
	Create a project in the Google Cloud Console.
	Generate OAuth 2.0 Credentials (Client ID and Client Secret).
	Register the authorized Callback URL (e.g., http://localhost:3000/auth/google/callback).
Configure passport-google-oauth20 Strategy:
	Instantiate the strategy using the Client ID, Client Secret, and Callback URL.
	Define the callback handler to extract user profile details (Google ID, email, name) and either find or create a user in your database.
Initiate Authentication (Redirection):
	Define a route (e.g., GET /auth/google) that calls passport.authenticate('google', { scope: ['profile', 'email'] }).
	Express redirects the user to Google’s OAuth consent screen.
Handle OAuth Callback & User Extraction:
	Define the callback route (e.g., GET /auth/google/callback) using passport.authenticate('google').
	Once the user authorizes access, Google redirects back to this endpoint with an authorization code, which Passport exchanges for user 	profile data before logging the user in.

Question 7 : Explain why we need middleware for security in Express. Write skeleton code
showing use of a security middleware (e.g. using helmet), plus custom middleware that
redirects HTTP to HTTPS (for production).
Answer: Express apps are susceptible to standard web vulnerabilities (e.g., Cross-Site Scripting, Clickjacking, MIME sniffing, and sensitive server-header leaks). Security middleware adds appropriate HTTP defense headers and enforces secure protocol rules dynamically across all incoming requests without needing manual implementation in every route.
import express from 'express';
import helmet from 'helmet';

const app = express();

// 1. Use Helmet to set secure HTTP headers
app.use(helmet());

// 2. Custom middleware to force HTTP -> HTTPS in production
app.use((req, res, next) => {
  if (process.env.NODE_ENV === 'production' && req.headers['x-forwarded-proto'] !== 'https') {
    return res.redirect(`https://${req.headers.host}${req.url}`);
  }
  next();
});

app.get('/', (req, res) => {
  res.send('Secure Express App');
});

app.listen(3000);

Question 8: What is NoSQL injection and why is it a threat in a MongoDB + Express app? Write
a small explanation AND describe a safe practice to avoid this risk when accepting user input
for queries.
Answer: NoSQL injection occurs when unsanitized user inputs containing MongoDB query operators (like $gt, $ne, or $regex) are passed directly into database queries.
In an Express app processing JSON requests, an attacker can pass an object like {"email": "admin@example.com", "password": {"$ne": null}} instead of a standard string password. MongoDB evaluates {"$ne": null} as "not equal to null", evaluating the query to true and bypassing authentication completely without knowing the password.

Safe Practices to Avoid NoSQL Injection:

	Sanitize Inputs: Use sanitization libraries like express-mongo-sanitize to automatically strip parameter keys starting with $ or 	containing ..

	Explicit Type Validation & Casting: Avoid passing req.body variables straight into queries. Explicitly cast inputs to strings or use 	schema validators (like Zod, Joi, or Mongoose schema definitions)
// UNSAFE:
User.findOne({ email: req.body.email, password: req.body.password });

// SAFE: Explicit string conversion
User.findOne({
  email: String(req.body.email),
  password: String(req.body.password)
});

Question 9: Your API serves a frontend hosted on another domain. What is CORS and why do
you need to configure it? Show minimal Express code to allow CORS only from
https://myfrontend.com.
Answer: CORS (Cross-Origin Resource Sharing) is a browser-implemented security mechanism governed by the Same-Origin Policy (SOP). By default, browsers block web applications on one domain from making HTTP requests to a server on a different domain, protocol, or port. You need to configure CORS on your Express backend to explicitly authorize trusted external domains (like [https://myfrontend.com](https://myfrontend.com)) to read response data from your API.
import express from 'express';
import cors from 'cors';

const app = express();

// Configure CORS to only allow requests from https://myfrontend.com
const corsOptions = {
  origin: 'https://myfrontend.com',
  optionsSuccessStatus: 200
};

app.use(cors(corsOptions));

app.get('/api/data', (req, res) => {
  res.json({ message: 'This response is accessible to https://myfrontend.com' });
});

app.listen(3000);

Question 10: Assume after login you store req.user.role as 'user' or 'admin'. Write a
middleware requireAdmin that only allows access if role is 'admin', else returns 403. Show
how to use it for route DELETE /admin/delete-user/:id.
Answer: import express from 'express';

const app = express();

// Authorization Middleware
const requireAdmin = (req, res, next) => {
  // Check if user is authenticated and has the 'admin' role
  if (req.user && req.user.role === 'admin') {
    return next();
  }

  return res.status(403).json({ error: 'Forbidden: Admin access required' });
};

// Protected Admin Route
app.delete('/admin/delete-user/:id', requireAdmin, (req, res) => {
  const userId = req.params.id;
  // Logic to delete user...
  res.json({ message: `User ${userId} successfully deleted.` });
});

app.listen(3000);

'''